In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import numpy as np
import io
import os
import sys
from PIL import Image
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from skimage.filters import gaussian, sobel
from skimage import measure, filters
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy import ndimage as ndi
from skimage.measure import label
from skimage.morphology import binary_opening
from skimage.segmentation import watershed

In [ ]:
# # with this peice of code, it will recognize the custom modules
# project_root = "/Users/cgeyskens/Documents/code/phd/image-analysis/synapse-counting"
# sys.path.append(project_root)

# custom modules
from synapse_counting import metadata, preprocessing, calc_synaptic_metrics, helpers, calc_synaptic_coloc

In [ ]:
# input folder
input_folder = "/Volumes/Intenso/image-analysis/synapse-counting/IHC_Exp12_VCAM1_VGAT-GEPH/airyscan_images"

# get a list of files in that input_folder
file_list = os.listdir(input_folder)
print(file_list)

protein_and_synaptic_marker = "VCAM1_LacZ_VGAT_GEPH"

In [ ]:
# standard filters
file = "/Volumes/Intenso/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA3_SR.czi"
pixel_size_um, _ , image_size_um = metadata.extract_metadata(file)
pre, post = preprocessing.extract_and_split(file, presynapse_channel = 0, postsynapse_channel = 1)
p = preprocessing.ImagePreprocessing(
            include_rolling_ball = True, radius = 10, # rolling ball parameters
            include_clahe = True, clip_limit = 0.005, kernel_size = 150, nbins = 265, # CLAHE parameters
            include_tophat = True, element_size = 5, # tophat parameters
            include_blur = True, sigma = 1, preserve_range = True # gaussian blur filters
            )
pre_1, post_1 = p.preprocess(pre, post)
presynapse_threshold = filters.threshold_otsu(pre_1)
presynapse_thresholded = pre_1 > presynapse_threshold

## Handcrafted preprocessing parameters

In [ ]:
preprocess_params_dict = {
    "VGLUT1_PSD95": {
        "CA1_SO": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 0.5,
            "puncta_size_threshold": 20, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.5,
            "post_threshold_scale": 0.7
        },
        "CA1_SR": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 0.5,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "otsu",
            "pre_threshold_scale": 1.6,
            "post_threshold_scale": 2
        },
        "CA1_SLM": {
            "include_rolling_ball": "True", "radius": 5,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 1,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.5,
            "post_threshold_scale": 0.7
        },
        "CA3_SO": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 0.5,
            "puncta_size_threshold": 20, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.5,
            "post_threshold_scale": 0.7
        },
        "CA3_SL": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 4,
            "puncta_size_threshold": 20, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.5,
            "post_threshold_scale": 1
        },
        "CA3_SR": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 4,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "otsu",
            "pre_threshold_scale": 2.3,
            "post_threshold_scale": 1.8
        },
        "DG_Hilus": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 4,
            "puncta_size_threshold": 20, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.5,
            "post_threshold_scale": 1
        },
        "DG_ML": {
            "include_rolling_ball": "True", "radius": 5,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 1,
            "puncta_size_threshold": 20,
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.4,
            "post_threshold_scale": 0.7
        }
    },

    "VGAT_GEPH": {
        "CA1_SO": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 1,
            "puncta_size_threshold": 2, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.3,
            "post_threshold_scale": 0.3
        },
        "CA1_SR": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 1,
            "puncta_size_threshold": 2, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.4,
            "post_threshold_scale": 0.3
        },
        "CA1_SLM": {
            "include_rolling_ball": "True", "radius": 5,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 1,
            "puncta_size_threshold": 2,
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.5,
            "post_threshold_scale": 0.6
        },
        "CA3_SO": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 1,
            "puncta_size_threshold": 5, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.5,
            "post_threshold_scale": 0.5
        },
        "CA3_SL": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 1,
            "puncta_size_threshold": 5, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.7,
            "post_threshold_scale": 0.8
        },
        "CA3_SR": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 1,
            "puncta_size_threshold": 5, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.3,
            "post_threshold_scale": 0.5
        },
        "DG_Hilus": {
            "include_rolling_ball": "True", "radius": 30,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 1,
            "puncta_size_threshold": 2, 
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.6,
            "post_threshold_scale": 0.6
        },
        "DG_ML": {
            "include_rolling_ball": "True", "radius": 5,
            "include_blur": "True", "sigma": 1,
            "include_clahe": "True",
            "include_tophat": "True", "element_size": 9,
            "watershed_sigma": 0.5,
            "puncta_size_threshold": 5,
            "threshold_algorithm": "yen",
            "pre_threshold_scale": 0.6,
            "post_threshold_scale": 0.6
        }

    }
}

In [ ]:
from skimage.filters import threshold_yen

# create synaptic_marker variable
synaptic_marker = helpers.get_synaptic_marker(protein_and_synaptic_marker)
print(synaptic_marker)
#  layer of interest
layers_of_interest = {"DG_ML"}

for filename in file_list:
    file_path = os.path.join(input_folder, filename)
    layer = helpers.get_hippocampal_layer(file_path)
    
    if layer in layers_of_interest:
        print(layer)
        print(filename)
        params = preprocess_params_dict.get(synaptic_marker, {}).get(layer, {})
        # extract metadata
        pixel_size_um, _ , image_size_um = metadata.extract_metadata(file_path)
        
        # extracting and splitting channels
        pre, post = preprocessing.extract_and_split(file_path, presynapse_channel = 0, postsynapse_channel = 1)
    
        # preprocessing
        p = preprocessing.ImagePreprocessing(
            include_rolling_ball=params["include_rolling_ball"], radius=params["radius"],
            include_blur=params["include_blur"], sigma = params["sigma"], preserve_range = True,
            include_clahe=params["include_clahe"],
            include_tophat=params["include_tophat"], element_size = params["element_size"]
        )
        pre_1, post_1 = p.preprocess(pre, post)

        # thresholding and watershed segmentation
        presynapse_threshold, postsynapse_threshold = preprocessing.thresholding(
            pre_1, 
            post_1, 
            threshold_algorithm=params["threshold_algorithm"],
            pre_threshold_scale=params["pre_threshold_scale"],
            post_threshold_scale=params["post_threshold_scale"]
            )

               
        presynapse_watersheded = preprocessing.custom_watershed(
            presynapse_threshold, 
            sigma = params["watershed_sigma"])
        
        postsynapse_watersheded = preprocessing.custom_watershed(
            postsynapse_threshold, 
            sigma = params["watershed_sigma"])
        
        _ , _ , _ , _ , _, pre_filtered, post_filtered= calc_synaptic_metrics.puncta_metrics(
            presynapse_watersheded, 
            postsynapse_watersheded, 
            image_size_um, 
            pixel_size_um, 
            puncta_size_threshold=params["puncta_size_threshold"])
        
    
        print("radius=", params["radius"], " element_size=", params["element_size"], " blur_sigma=" ,  params["sigma"] ,  " watershed_sigma=" , params["watershed_sigma"]) 
       
        # Display the pre-processed and post-processed images
        fig, axes = plt.subplots(4, 2, figsize=(12, 16))

        axes[0, 0].imshow(pre, cmap='gray')
        axes[0, 0].set_title('Original Presynapse Image')
        axes[0, 0].axis('off')

        axes[0, 1].imshow(post, cmap='gray')
        axes[0, 1].set_title('Original Postsynapse Image')
        axes[0, 1].axis('off')

        axes[1, 0].imshow(pre_1, cmap='gray')
        axes[1, 0].set_title('Pre-processed Presynapse Image')
        axes[1, 0].axis('off')

        axes[1, 1].imshow(post_1, cmap='gray')
        axes[1, 1].set_title('Pre-processed Postsynapse Image')
        axes[1, 1].axis('off')

        axes[2, 0].imshow(presynapse_watersheded, cmap='gray')
        axes[2, 0].set_title('Binarized Presynapse Image')
        axes[2, 0].axis('off')

        axes[2, 1].imshow(postsynapse_watersheded, cmap='gray')
        axes[2, 1].set_title('Binarized Postsynapse Image')
        axes[2, 1].axis('off')

        axes[3, 0].imshow(pre_filtered, cmap='gray')
        axes[3, 0].set_title('Binarized Presynapse Image with filter')
        axes[3, 0].axis('off')

        axes[3, 1].imshow(post_filtered, cmap='gray')
        axes[3, 1].set_title('Binarized Postsynapse Image with filter')
        axes[3, 1].axis('off')


        plt.tight_layout()
        plt.show()

    

## Handcrafted parameter ranges



In [ ]:
params_ranges_dict = {
    "VGLUT1_PSD95": {
        "CA1_SO": {
            "pre_distance": [1, 5], 
            "post_distance": [1, 5], 
            "pre_threshold": [500, 600], 
            "post_threshold": [100, 150], 
            "max_distance_um": [0.01, 1]
        },
        "CA1_SR": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [300, 600],
            "post_threshold": [100, 300],
            "max_distance_um": [0.01, 1]
        },
        "CA1_SLM": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [250, 350],
            "post_threshold": [100, 200],
            "max_distance_um": [0.01, 1]
        },
        "CA3_SO": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [500, 1200],
            "post_threshold": [100, 160],
            "max_distance_um": [0.01, 1]
        },
        "CA3_SL": {
            "pre_distance": [10, 15],
            "post_distance": [5, 10],
            "pre_threshold": [1000, 2000],
            "post_threshold": [250, 400],
            "max_distance_um": [0.01, 1]
        },
        "CA3_SR": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [500, 1000],
            "post_threshold": [100, 200],
            "max_distance_um": [0.01, 1]
        },
        "DG_Hilus": {
            "pre_distance": [5, 15],
            "post_distance": [1, 10],
            "pre_threshold": [1000, 3000],
            "post_threshold": [300, 500],
            "max_distance_um": [0.01, 1]
        },
        "DG_ML": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [500, 1000],
            "post_threshold": [100, 200],
            "max_distance_um": [0.01, 1]
        }
    }, 
    
    "VGAT_GEPH": {
        "CA1_SO": {
            "pre_distance": [1, 5], 
            "post_distance": [1, 5], 
            "pre_threshold": [300, 450], 
            "post_threshold": [250, 400], 
            "max_distance_um": [0.01, 1]
        },
        "CA1_SR": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [300, 450],
            "post_threshold": [250, 400],
            "max_distance_um": [0.01, 1]
        },       
        "CA1_SLM": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [200, 350],
            "post_threshold": [200, 300],
            "max_distance_um": [0.01, 1]
        },
        "CA3_SO": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [300, 500],
            "post_threshold": [300, 500],
            "max_distance_um": [0.01, 1]
        },
        "CA3_SL": {
            "pre_distance": [10, 15],
            "post_distance": [5, 15],
            "pre_threshold": [500, 1000],
            "post_threshold": [500, 800],
            "max_distance_um": [0.01, 1]
        },
        "CA3_SR": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [300, 450],
            "post_threshold": [250, 400],
            "max_distance_um": [0.01, 1]
        },
        "DG_Hilus": {
            "pre_distance": [5, 15],
            "post_distance": [5, 15],
            "pre_threshold": [400, 1000],
            "post_threshold": [400, 1000],
            "max_distance_um": [0.01, 1]
        },
        "DG_ML": {
            "pre_distance": [1, 5],
            "post_distance": [1, 5],
            "pre_threshold": [300, 400],
            "post_threshold": [300, 400],
            "max_distance_um": [0.01, 1]
        }            
    }
}

In [ ]:
# standard filters
synaptic_marker = helpers.get_synaptic_marker(protein_and_synaptic_marker)
layer = "CA1_SLM"
file = "/Volumes/Intenso/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp4_IHC-Exp2_Brain-4_section-3_488-VGLUT1_647-PSD95_LacZ-gRNA_63X&3XzoomAiryscan_CA3_SL.czi"
pixel_size_um, _ , image_size_um = metadata.extract_metadata(file)
pre_LacZ, post_LacZ = preprocessing.extract_and_split(file, presynapse_channel = 0, postsynapse_channel = 1)
params = preprocess_params_dict.get(synaptic_marker, {}).get(layer, {})
p = preprocessing.ImagePreprocessing(
            include_rolling_ball=params["include_rolling_ball"], radius=params["radius"],
            include_blur=params["include_blur"], sigma = params["sigma"], preserve_range = True,
            include_clahe=False,
            include_tophat=params["include_tophat"], element_size = params["element_size"]
            )
pre_1_lacZ, post_1_LacZ = p.preprocess(pre_LacZ, post_LacZ)


In [ ]:
# standard filters
file = "/Volumes/Intenso/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/CRISPR-Exp5_IHC-Exp2_Brain-5_section-1_488-VGLUT1_647-PSD95_VCAM1-gRNA_63X&3XzoomAiryscan_CA3_SR.czi"
pixel_size_um, _ , image_size_um = metadata.extract_metadata(file)
pre_VCAM1, post_VCAM1 = preprocessing.extract_and_split(file, presynapse_channel = 0, postsynapse_channel = 1)
params = preprocess_params_dict.get(synaptic_marker, {}).get(layer, {})
p = preprocessing.ImagePreprocessing(
            include_rolling_ball=params["include_rolling_ball"], radius=params["radius"],
            include_blur=params["include_blur"], sigma = params["sigma"], preserve_range = True,
            include_clahe=False,
            include_tophat=params["include_tophat"], element_size = params["element_size"]
            )
pre_1_VCAM1, post_1_VCAM1 = p.preprocess(pre_VCAM1, post_VCAM1)

In [ ]:
import czifile
import os
from lxml import etree
from synapse_counting import results_plotting

czi = czifile.CziFile(file)
czi_xml_str = czi.metadata() # gets the metadata in a xml string format
czi_parsed = etree.fromstring(czi_xml_str) # parses the czi_xml_str file

In [ ]:
print(file)
print(czi_xml_str)

In [ ]:
superres_params = czi_parsed.findall(".//SuperResolutionParameter")

ch_1_superresparam_value = float(superres_params[0].text)
ch_2_superresparam_value = float(superres_params[1].text)


print(ch_1_superresparam_value, ch_2_superresparam_value )

In [ ]:
# making a loop to get the supperres_params for each image for VCAM1, VGLUT1-PSD95 condition

# input folder
input_folder = "/Volumes/Intenso/image-analysis/synapse-counting/VCAM1/VCAM1-LacZ_VGLUT1-PSD95_images/"

# get a list of files in that input_folder
file_list = os.listdir(input_folder)

print(file_list)

superres_params_list = []

for file in file_list:
    long_file = input_folder + file
    czi = czifile.CziFile(long_file)
    czi_xml_str = czi.metadata()
    czi_parsed = etree.fromstring(czi_xml_str)
    superres_params = czi_parsed.findall(".//SuperResolutionParameter")

    ch_1_superresparam_value = float(superres_params[0].text)
    ch_2_superresparam_value = float(superres_params[1].text)

    superres_params_list.append({"img_filename": file, "ch_1_superresparam_value": ch_1_superresparam_value, "ch_2_superresparam_value": ch_2_superresparam_value})

print(superres_params_list)


superres_params_df = pd.DataFrame(superres_params_list)
superres_params_df

In [ ]:
superres_params_df["section"] =superres_params_df['img_filename'].str.replace(r'_[^_]+-gRNA_', '_', regex=True)
superres_params_df["gRNA"] = superres_params_df['img_filename'].apply(lambda x: x.split('_')[-4:-3]).apply(lambda x: '_'.join(x))


superres_params_df

In [ ]:
df = superres_params_df

In [ ]:
df_ordered = df.pivot_table(index='section', columns = "gRNA", values = ["ch_1_superresparam_value", "ch_2_superresparam_value"])
df_ordered.head(10)

In [ ]:
df_ordered["diff_ch1"] = df_ordered[('ch_1_superresparam_value',  'LacZ-gRNA')] - df_ordered[('ch_1_superresparam_value', 'VCAM1-gRNA')]
df_ordered["diff_ch2"] = df_ordered[('ch_2_superresparam_value',  'LacZ-gRNA')] - df_ordered[('ch_2_superresparam_value', 'VCAM1-gRNA')]
df_ordered

In [ ]:
df_ordered = df_ordered.reset_index().rename(columns={'index': 'section'})
df_ordered['hippocampal_layer'] = df_ordered['section'].apply(lambda x: x.split('_')[-2:]).apply(lambda x: ' '.join(x))
df_ordered

In [ ]:
sns.catplot(
    data=df_ordered, 
    x="hippocampal_layer", 
    y="diff_ch1",
    height=5,
    aspect=2
).set(
    title="VGLUT1 (ch1) super-resolution-param-differences [LacZ-gRNA - VCAM1-gRNA]"
)

In [ ]:
sns.catplot(
    data=df_ordered, 
    x="hippocampal_layer", 
    y="diff_ch2",
    height=5,
    aspect=2
).set(
    title="PSD95 (ch2) super-resolution-param-differences [LacZ-gRNA - VCAM1-gRNA]"
)


In [ ]:
%gui qt5
import napari
from skimage import data

# Sample images
image1 = data.camera()
image2 = data.coins()

# First viewer
viewer1 = napari.Viewer()
viewer1.add_image(pre_1_lacZ, name='pre_1_lacZ')

# Second viewer
viewer2 = napari.Viewer()
viewer2.add_image(pre_1_VCAM1, name='pre_1_VCAM1')


In [ ]:
# create synaptic_marker variable
synaptic_marker = helpers.get_synaptic_marker(protein_and_synaptic_marker)
print(synaptic_marker)
#  layer of interest
layers_of_interest = {"CA3_SR"}

for filename in file_list:
    file_path = os.path.join(input_folder, filename)
    layer = helpers.get_hippocampal_layer(file_path)
    if layer in layers_of_interest:
        print(filename)
        params = preprocess_params_dict.get(synaptic_marker, {}).get(layer, {})

        # extract metadata
        pixel_size_um, _ , image_size_um = metadata.extract_metadata(file_path)
        
        # extracting and splitting channels
        pre, post = preprocessing.extract_and_split(file_path, presynapse_channel = 0, postsynapse_channel = 1)
    
        # preprocessing
        p = preprocessing.ImagePreprocessing(
        include_rolling_ball=params["include_rolling_ball"], radius=params["radius"],
        include_blur=params["include_blur"], sigma = params["sigma"], preserve_range = True,
        include_clahe=False,
        include_tophat=params["include_tophat"], element_size = params["element_size"]
        )
        pre_1, post_1 = p.preprocess(pre, post) 

        pre_coord, post_coord, _ = calc_synaptic_coloc.local_peak_detection(presynapse_preprocessed = pre_1, 
                                                        postsynapse_preprocessed = post_1, 
                                                        presynapse_distance = 3, 
                                                        postsynapse_distance = 3, 
                                                        presynapse_threshold = 999.48, 
                                                        postsynapse_threshold = 192.27, 
                                                        plot_coord = False)
        
        fig, axs = plt.subplots(2, 2, figsize=(30, 30))

        axs[0,0].imshow(pre_1, cmap='gray')
        #axs[0,0].plot(presynapse_coord[:, 1], presynapse_coord[:, 0], 'c.')
        axs[0,0].set_title('pre_preprocessed')

        axs[0,1].imshow(post_1, cmap='gray')
        #axs[0,1].plot(postsynapse_coord[:, 1], postsynapse_coord[:, 0], 'm.')
        axs[0,1].set_title('post_preprocessed')

        axs[1,0].imshow(pre_1, cmap='gray')
        axs[1,0].plot(pre_coord[:, 1], pre_coord[:, 0], 'c.')
        axs[1,0].set_title('pre_preprocessed_with_local_peaks')

        axs[1,1].imshow(post_1, cmap='gray')
        axs[1,1].plot(post_coord[:, 1], post_coord[:, 0], 'm.')
        axs[1,1].set_title('post_preprocessed_with_local_peaks')

        plt.show()